In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Note] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 21. RLHF — Reinforcement Learning from Human Feedback ⭐ [CPU/GPU Benchmark ⑨]

> **Learning Goals**
> - Map basic reinforcement learning concepts (MDP, policy, reward, value) to LLMs
> - Derive policy gradients and REINFORCE
> - Understand and implement PPO's clipped objective
> - Explain reward model (RM) training

## 21.1 Limits of SFT and the Alignment Problem

An SFT model has learned the response format, but it does not know whether humans **prefer** its answers. It may generate harmful or incorrect but plausible responses. Alignment means adapting the model to human values.

## 21.2 Three Steps of RLHF

1. **SFT**: instruction following (Ch 20)
2. **Reward Model (RM)**: train a model that scores response pairs
3. **PPO**: use the RM as a reward signal to reinforce the policy (= LLM)

## 21.3 RL Basics — MDP

**MDP** (Markov Decision Process): $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$
- $\mathcal{S}$: state space
- $\mathcal{A}$: action space
- $P(s'|s, a)$: transition probability
- $R(s, a)$: reward
- $\gamma$: discount factor

LLM mapping:
- state $s_t$ = generated tokens so far $[w_0, \ldots, w_{t-1}]$
- action $a_t$ = next token $w_t$
- policy $\pi_\theta(a_t | s_t) = P(w_t | w_{<t}; \theta)$
- reward $R$ = score assigned by the RM, often only at the final token


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# vocabulary size (toy)
vocab_size = 100  # used throughout this chapter
print(f"vocab_size (toy): {vocab_size}")


## 21.4 Policy Gradient Theorem (REINFORCE)

Goal: maximize $J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]$.

$$\nabla J(\theta) = \mathbb{E}_{\tau} \left[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\right]$$

- $G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$: discounted return
- $\nabla \log \pi$: gradient of the log probability, controlling how likely the action becomes
- Weighting by $G_t$ increases probabilities for actions that lead to good rewards

**Advantage** $A_t = G_t - V(s_t)$ subtracts a baseline $V$ to reduce variance.


In [ ]:
# REINFORCE example (simple toy MDP)
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Environment: simple bandit with 3 arms and different reward distributions
true_rewards = [0.2, 0.5, 0.8]  # arm 2 is best

class PolicyNet(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.fc = nn.Linear(1, 16)
        self.head = nn.Linear(16, n_actions)
    def forward(self, x):
        return self.head(F.relu(self.fc(x)))

policy = PolicyNet(3)
opt = torch.optim.Adam(policy.parameters(), lr=0.01)

n_episodes = 500
all_rewards = []
for ep in range(n_episodes):
    # episode: 1 step (bandit)
    state = torch.tensor([[1.0]])
    logits = policy(state)
    probs = F.softmax(logits, dim=-1)
    action = torch.multinomial(probs, 1).item()
    reward = true_rewards[action] + np.random.randn() * 0.05

    # REINFORCE: log_prob * reward
    log_prob = F.log_softmax(logits, dim=-1)[0, action]
    loss = -log_prob * reward  # maximize by minimizing the negative
    opt.zero_grad()
    loss.backward()
    opt.step()
    all_rewards.append(reward)

print(f"Mean reward over first 50 episodes: {np.mean(all_rewards[:50]):.3f}")
print(f"Mean reward over last 50 episodes: {np.mean(all_rewards[-50:]):.3f}")
print(f"Theory Maximum: {max(true_rewards):.3f}")


## 21.5 PPO — Trust Region and Clipped Objective

REINFORCE has high variance. PPO prevents the policy from changing too much in one update.

**Ratio**: $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$

**Clipped Objective**:
$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[\min\left(r_t \hat{A}_t, \, \mathrm{clip}(r_t, 1-\epsilon, 1+\epsilon) \hat{A}_t\right)\right]$$

- $\epsilon \approx 0.2$: allowed ratio range
- Clipping with `min` prevents overly large policy updates
- This approximates a trust region


In [ ]:
# PPO example (toy problem)
import torch
import torch.nn as nn
import torch.nn.functional as F

class Policy(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.fc = nn.Linear(4, 32)
        self.head = nn.Linear(32, n_actions)
    def forward(self, x):
        return self.head(F.tanh(self.fc(x)))

# simple environment (random)
def env_step(state, action):
    reward = float((state.sum() + action - 2) / 10)  # simple reward
    next_state = state + torch.randn(4) * 0.1
    return next_state, reward, False

policy = Policy(3)
opt = torch.optim.Adam(policy.parameters(), lr=3e-4)

# PPO update
def ppo_update(policy, opt, states, actions, old_log_probs, advantages, returns, values,
               epsilon=0.2, n_epochs=4):
    for _ in range(n_epochs):
        logits = policy(states)
        new_log_probs = F.log_softmax(logits, dim=-1)
        new_log_probs_a = new_log_probs.gather(1, actions.unsqueeze(1)).squeeze()

        ratio = torch.exp(new_log_probs_a - old_log_probs)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
        actor_loss = -torch.min(surr1, surr2).mean()

        # value function loss (simple)
        critic_loss = F.mse_loss(values.squeeze(), returns)

        loss = actor_loss + 0.5 * critic_loss
        opt.zero_grad()
        loss.backward()
        opt.step()

# training simulation
n_iterations = 50
all_rewards = []
for it in range(n_iterations):
    # rollout collection
    states, actions, rewards, old_log_probs = [], [], [], []
    state = torch.randn(1, 4)
    ep_reward = 0
    for step in range(20):
        with torch.no_grad():
            logits = policy(state)
            probs = F.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            old_log_prob = dist.log_prob(action)
        next_state, reward, done = env_step(state.squeeze(), action.item())
        states.append(state.squeeze())
        actions.append(action)
        rewards.append(reward)
        old_log_probs.append(old_log_prob)
        ep_reward += reward
        state = next_state.unsqueeze(0)
    all_rewards.append(ep_reward / 20)

    # advantage calculation (simple: use reward directly)
    advantages = torch.tensor(rewards)
    returns = torch.tensor(rewards)
    values = torch.zeros_like(returns)  # no baseline (simplified)

    ppo_update(policy, opt, torch.stack(states), torch.tensor(actions),
               torch.stack(old_log_probs), advantages, returns, values)

print(f"Mean reward over first 10 episodes: {np.mean(all_rewards[:10]):.4f}")
print(f"Mean reward over last 10 episodes: {np.mean(all_rewards[-10:]):.4f}")


## 21.6 Reward Model (RM) Training

An RM scores responses. It is trained on **preference data** (chosen vs rejected).

**Bradley-Terry model**:
$$P(\mathbf{y}_w \succ \mathbf{y}_l | \mathbf{x}) = \sigma(r(\mathbf{x}, \mathbf{y}_w) - r(\mathbf{x}, \mathbf{y}_l))$$

Loss (NLL):
$$\mathcal{L}_{\text{RM}} = -\log \sigma(r(\mathbf{x}, \mathbf{y}_w) - r(\mathbf{x}, \mathbf{y}_l))$$

- $\mathbf{y}_w$: preferred (chosen) response
- $\mathbf{y}_l$: rejected response
- $r(\mathbf{x}, \mathbf{y})$: RM score

Architecture: add a scalar output head to the SFT model.


In [ ]:
# RM training simulation
import torch
import torch.nn as nn
import torch.nn.functional as F

class RewardModel(nn.Module):
    def __init__(self, vocab_size, d_model=32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.rnn = nn.LSTM(d_model, d_model, batch_first=True)
        self.head = nn.Linear(d_model, 1)  # scalar reward

    def forward(self, x):
        emb = self.emb(x)
        _, (h, _) = self.rnn(emb)
        return self.head(h.squeeze(0))  # (B,)

# toy preference data
n_samples = 100
torch.manual_seed(0)
rm = RewardModel(vocab_size=vocab_size, d_model=32)
opt = torch.optim.AdamW(rm.parameters(), lr=1e-3)

# random chosen/rejected pairs (simulation)
print("RM Training simulation:")
losses = []
for step in range(200):
    # synthetic data: short sequences
    chosen = torch.randint(0, vocab_size, (8, 16))
    rejected = torch.randint(0, vocab_size, (8, 16))
    # artificial difference: assume chosen sequences have smaller mean IDs
    # in practice, this is the pattern the RM would learn
    r_chosen = rm(chosen).squeeze(-1)
    r_rejected = rm(rejected).squeeze(-1)
    loss = -F.logsigmoid(r_chosen - r_rejected).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.plot(losses)
plt.xlabel('Step'); plt.ylabel('RM Loss')
plt.title('Reward Model Training (Bradley-Terry)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch21_rm_training.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Final RM loss: {losses[-1]:.4f} (theoretical random baseline ≈ 0.69 = -log(0.5))")


## 21.7 KL Penalty — Stay Close to the Reference Model

If RLHF maximizes only the RM score, the model can generate strange text that exploits the reward model (reward hacking).

Solution: add a KL penalty so the policy stays close to the reference model (usually the SFT model).

$$R_{\text{total}} = R_{\text{RM}}(\mathbf{x}, \mathbf{y}) - \beta \, D_{\text{KL}}(\pi_\theta(\cdot|\mathbf{x}) \| \pi_{\text{ref}}(\cdot|\mathbf{x}))$$

- $\beta$: KL strength, commonly around 0.01-0.5
- $\pi_{\text{ref}}$: fixed SFT model
- Token-level KL: $\sum_t \log \frac{\pi_\theta(w_t|...)}{\pi_{\text{ref}}(w_t|...)}$


In [ ]:
# KL divergence calculation (policy vs reference)
def token_kl_div(policy_logits, ref_logits):
    """token-wise KL Divergence."""
    p = F.softmax(policy_logits, dim=-1)
    log_p = F.log_softmax(policy_logits, dim=-1)
    log_q = F.log_softmax(ref_logits, dim=-1)
    return (p * (log_p - log_q)).sum(dim=-1)  # (B, T)

# simulation
torch.manual_seed(0)
B, T, V = 4, 16, 100
ref_logits = torch.randn(B, T, V)
policy_logits = ref_logits + torch.randn(B, T, V) * 0.5  # slightly different policy

kl = token_kl_div(policy_logits, ref_logits)
print(f"KL divergence shape: {kl.shape}")
print(f"Mean KL: {kl.mean():.4f}")
print(f"Maximum KL: {kl.max():.4f}")
print("\n=> As the policy moves farther from the reference, KL increases. RLHF uses this as a penalty.")


## 21.8 [CPU/GPU Benchmark ⑨] SFT vs RLHF Training Comparison

RLHF requires four models: policy, reference, RM, and value function. Memory and time cost are roughly 3-4x higher.


In [ ]:
# RLHF memory analysis (simulation)
print("Models required for RLHF training:")
print("  1. Policy model (trainable)")
print("  2. Reference model (frozen, for KL)")
print("  3. Reward model (frozen)")
print("  4. value/critic model (trainable, optional)")
print()

# example: 7B LLaMA RLHF
model_size_gb = 7 * 2 * 4  # 7B params * 2 bytes (FP16) * 4 models? No:
# In practice: policy uses FP32 gradients and FP32 optimizer state
# AdamW: param + grad + m + v = 4x
# 7B model FP16: 14GB
# AdamW state FP32: 7B * 8 bytes = 56GB
# policy: 14 + 56 = 70GB
# reference: 14GB
# RM: 14GB
# value: 14 + 56 = 70GB
# total: about 168GB → A100 80GB 2 required

print("7B Model RLHF Memory Estimation:")
print(f"  Policy (FP16 + AdamW FP32): ~70GB")
print(f"  Reference (FP16, frozen): ~14GB")
print(f"  Reward Model (FP16, frozen): ~14GB")
print(f"  values (FP16 + AdamW): ~70GB")
print(f"  total: ~168GB (A100 80GB × 2 required)")
print("\n=> RLHF very expensive. DPO (Ch 22) is a more efficient alternative.")


## 21.9 Pitfalls of RLHF

- **Reward hacking**: high RM score but actually poor responses
- **Mode collapse**: loss of diversity, repeatedly producing one pattern
- **Instability**: PPO is sensitive to hyperparameters

## 21.10 Key Takeaways

| Concept | Formula | Meaning |
|---|---|---|
| Policy gradient | $\nabla J = \mathbb{E}[\nabla\log\pi \cdot G]$ | REINFORCE |
| Advantage | $A = G - V$ | remove baseline |
| PPO clip | $\min(rA, \text{clip}(r)A)$ | trust region |
| RM loss | $-\log\sigma(r_w - r_l)$ | Bradley-Terry |
| KL penalty | $R - \beta D_{KL}$ | keep reference behavior |

## Exercises

1. Train a toy environment such as CartPole with REINFORCE.
2. Compare PPO with $\epsilon = 0.1, 0.2, 0.3$ and analyze the effect.
3. Compare RM loss when chosen and rejected responses are similar vs very different.
4. Simulate policy changes under KL penalty $\beta = 0, 0.1, 1.0$.
5. Explain the memory advantage of DPO over RLHF.

> Solutions: `solutions/ch21_solutions.ipynb`
